# Lab type: review
# Course: ML203 — Unsupervised Learning & Clustering
# Lesson: t-SNE and UMAP
# Task: Evaluate the visualizations from t-SNE and UMAP. Answer the judgment questions.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

## Step 1: Load Data (t-SNE and UMAP are slow, so we'll use a smaller dataset)

In [ ]:
# Load the digits dataset (1797 images of hand-written digits, 64 features)
digits = load_digits()
X = digits.data
y = digits.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Data shape: {X_scaled.shape}")
print(f"Classes (0-9): {np.unique(y)}")


## Step 2: Review PCA Visualization (Linear Baseline)

In [ ]:
# Review this code — what does PCA tell us about the digit structure?
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='tab10', alpha=0.6)
plt.colorbar(scatter, label='Digit')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.title('PCA Visualization')
plt.show()

**Question:** Looking at the PCA plot, are the digit classes well-separated? What does this tell you about whether digits have distinct high-level structure?

<details>
<summary>🔑 Reveal answer — Q1</summary>

**Separation in PCA plot:** Partial — some digit classes form loose clusters, but significant overlap exists. This tells you that digits have real linear structure (otherwise PCA would show a uniform blob), but the top two principal components explain only a fraction of total variance, so the full class discrimination cannot be recovered from this 2D projection alone.

**What this means:** Digit differences are not purely linear; at least some class boundaries require nonlinear separation. This is precisely why t-SNE and UMAP improve the picture — they can capture relationships that a linear projection misses.

</details>

## Step 3: Review t-SNE Visualization

In [ ]:
# Review this code — how does t-SNE differ from PCA?
# Note: t-SNE is slow. We'll sample data to keep it manageable.

try:
    from sklearn.manifold import TSNE
    # Sample to speed up t-SNE
    sample_idx = np.random.choice(X_scaled.shape[0], 500, replace=False)
    X_sample = X_scaled[sample_idx]
    y_sample = y[sample_idx]
    
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
    X_tsne = tsne.fit_transform(X_sample)
    
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_sample, cmap='tab10', alpha=0.6)
    plt.colorbar(scatter, label='Digit')
    plt.title('t-SNE Visualization (500 samples)')
    plt.show()
    
except ImportError:
    print("t-SNE requires sklearn; it's available in scikit-learn >= 0.17")

**Question:** How does the t-SNE plot differ from PCA? Are digits more separated? What does this tell you about the structure of the data?

<details>
<summary>🔑 Reveal answer — Q2</summary>

**How t-SNE differs:** t-SNE optimises a nonlinear objective that pulls similar points together and pushes dissimilar points apart. The result is tight, well-separated islands for each digit class — dramatically better local separation than PCA.

**Critical caveat:** Distances *between* clusters in the t-SNE plot are not meaningful. You cannot infer that digit 4 is "twice as different" from digit 7 as digit 1 based on plot geometry — the inter-cluster spacing is an artefact of the optimisation, not a distance measure.

**Reliability:** t-SNE output varies with `perplexity` and random seed. Always treat clusters as qualitative structure and validate findings with a metric (silhouette score, classification accuracy) before drawing conclusions.

</details>

## Step 4: Review UMAP Visualization

In [ ]:
# Review this code — how does UMAP compare to t-SNE?
# Note: UMAP must be installed separately: pip install umap-learn

try:
    from umap import UMAP
    
    # Use the same sample as t-SNE for fair comparison
    umap = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    X_umap = umap.fit_transform(X_sample)
    
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(X_umap[:, 0], X_umap[:, 1], c=y_sample, cmap='tab10', alpha=0.6)
    plt.colorbar(scatter, label='Digit')
    plt.title('UMAP Visualization (500 samples)')
    plt.show()
    
except ImportError:
    print("UMAP must be installed: pip install umap-learn")

**Question:** How does UMAP differ from t-SNE? Which better preserves the global structure? Which better preserves local structure?

<details>
<summary>🔑 Reveal answer — Q3</summary>

**UMAP vs t-SNE:** UMAP preserves both local *and* global structure better than t-SNE. Clusters remain tight (local), and the relative positions of cluster groups tend to be more stable and meaningful across runs (global). t-SNE sacrifices global structure to maximise local separation — its cluster layout can change substantially across perplexity values or random seeds.

**Speed:** UMAP is substantially faster, especially on large datasets.

**Practical guidance:** Use UMAP as the default for high-dimensional visualisation. Use t-SNE only when you specifically need the tightest possible local cluster separation and can afford to tune perplexity carefully. Whichever you use, validate any apparent structure with quantitative metrics — neither algorithm guarantees that visual clusters correspond to real classes.

</details>

## Step 5: Important Caveat About These Visualizations

In [ ]:
print("IMPORTANT: These visualizations are for exploration, not truth.")
print("\nWhat can go wrong:")
print("- t-SNE and UMAP can create structure that doesn't exist in the original data")
print("- Two clusters that look separate in t-SNE might overlap in the original high-dimensional space")
print("- Perplexity and n_neighbors choices affect the results significantly")
print("\nUse these tools to:")
print("1. Explore and get intuition about data")
print("2. Spot obvious outliers or obvious clusters")
print("\nDo NOT use these plots to:")
print("1. Make final decisions about cluster count or structure")
print("2. Trust the exact distances or separations shown")
print("3. Replace rigorous clustering validation metrics")

<details>
<summary>🔑 Reveal summary answers</summary>

1. **PCA:** Shows partial linear separation — useful as a fast baseline, but misses nonlinear structure between digit classes.
2. **t-SNE:** Strong local cluster separation; inter-cluster distances are not interpretable; output varies with hyperparameters.
3. **UMAP vs t-SNE:** UMAP preserves global structure better and runs faster — prefer it as the default; use t-SNE when tight local separation is the priority.

</details>

## Summary

t-SNE and UMAP are powerful visualization tools:
- **PCA:** Linear, preserves global structure, fast
- **t-SNE:** Nonlinear, emphasizes local structure, slower, prone to artifacts
- **UMAP:** Nonlinear, better global structure than t-SNE, faster than t-SNE

**Use for exploration, not decisions.** Always validate findings with rigorous metrics.

**Next lesson:** Evaluating Clustering — how to assess cluster quality when you have no ground truth.